In [ ]:
from amplpy import AMPL
ampl = AMPL()

# Overview
This Jupyter notebook serves as documentation for the Amplify library.  A native AMPL implementation breaks the library into three components:
1. A model file where the library and user code reside,
2. A data file where the values of parameters, variables, and sets reside, and
3. A run file, where the problem set up and solution occur.

In this notebook, the three files are intermingled to help make the experience more interactice.  In general, code prefixed with a `data;` command are meant to go into an AMPL data (.dat) file.  The next code cell and the final two code cells are meant to go into the run (.run) file.  All other code cells are part of the model (.mod) file.

# Amplify Script
The first lines of an Amplify script file is where we reset the AMPL interpreter and set various AMPL and solver options.

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* settings
#----------- clear and reload
reset;
#----------- options
option presolve_eps 1e-14;
option solver ipopt;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* settings!

# Grid Model Section
The grid module defines 

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
#----------- modules
# model-defined modules;
set MOD_MODULES;
set MOD_GRID {MOD_MODULES}; # this set will be redeclared later

# data-defined modules
set DAT_MODULES default {};
set DAT_GRID {DAT_MODULES};

# all modules
set MODULES = MOD_MODULES union DAT_MODULES;
set GRID_MODULES {i1 in MODULES} := 
  if i1 in MOD_MODULES then MOD_GRID[i1]  
  else DAT_GRID[i1];

#----------- phases
# dynamics are constant within a phase
set SUBPHASES within Reals;
set SUBPHASES_MBRS = {'PHASE', 'BEG', 'END'};
param subphases {SUBPHASES, SUBPHASES_MBRS};

# subphases must map to phase through floor function
check {i1 in SUBPHASES} : subphases[i1, 'PHASE'] = floor(i1);
# beginning must be less than end
check {i1 in SUBPHASES} : subphases[i1, 'BEG'] <= subphases[i1, 'END'];
# make sure sets don't overlap
check {i1 in SUBPHASES, i2 in SUBPHASES : i1 != i2} : inter {i3 in {i1, i2}} union {i4 in {'BEG', 'END'}} {subphases[i3, i4]} within {};

set PHASES = setof {i1 in SUBPHASES} subphases[i1, 'PHASE'];

#----------- grid points
param GRID_MAX >= 0;

set GRID_MBRS = {0} union SUBPHASES union PHASES;

set GRID {i1 in GRID_MBRS} ordered by [0, GRID_MAX] :=
  if i1 = 0 then # add grid points from registered modules
    union {i2 in MODULES} GRID_MODULES[i2]
  else if i1 in Integers then # collect grid points for each phase
    union {i2 in SUBPHASES : floor(i2) = i1} GRID[i2]
  else {i2 in GRID[0] : subphases[i1, 'BEG'] <= i2 <= subphases[i1, 'END']};  # collect grid points for each subphase

#----------- decision variable: time
var t {GRID[0]}; # time
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

# Grid Data Section

## About the Example
In the example below, 26 grid point will be created from 0 to 25.  There is one subphase with label `1.1`.  It has the same equations of motion as other subphases in phase 1.  The subphase is defined across grid points 0 to 25, i.e., the entire grid.

## Details
`GRID_MAX`: the maximum number of grid points

`subphases`: a list of subphase labels

`subphase`: a 2D table mapping subphases to grid points
- `1.1`: 1.1 is a user-defined subphase label; must be a real number such that phase = floor(subphase label)
- `PHASE`: the phase the subphase is part of
- `BEG`: the start grid point in the interval
- `END`: the last consecutive grid point in the interval

A subphase in Amplify is a continuous interval of grid points, from `BEG` to `END` with the same constrained equations of motion.  In fact, subphases $i$ and $j$ such that 

>  floor(subphase label $i$) == floor(subphase label $j$)

have the same constrained equations of motion.  Users can further customize a subphase with additional variables and constraints without affecting other subphases.


**CAUTION**: `PHASE` will likely be removed as Amplify assumes that `PHASE` = floor(subphase label)

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# params
param GRID_MAX := 25.;
set SUBPHASES := 1.1;
param subphases := [*,:]
     :PHASE   BEG   END :=
1.1   1.       0.    25.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!
model;

In [ ]:
%%ampl_eval

#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* bezier
# Resources:
# https://pages.mtu.edu/~shene/COURSES/cs3621/NOTES/spline/Bezier/bezier-der.html

set BEZ_LBLS;
set BEZ_SUBGRID {BEZ_LBLS};
set BEZ_MBRS := {'DER', 'CP_ROW', 'CP_COL'};
set BEZ_SUBGRID_MBRS := {'BEG', 'END'};

param bez {BEZ_LBLS, BEZ_MBRS};
param bez_subgrid {i1 in BEZ_LBLS, BEZ_SUBGRID[i1], BEZ_SUBGRID_MBRS};

set BEZ {i1 in BEZ_LBLS, i2 in BEZ_MBRS} :=
  if i2 in {'DER', 'DEG'} then 0..bez[i1, i2]
  else if i2 in {'CP_ROW', 'CP_COL'} then 0..bez[i1, i2]-1
  else {};

set BEZ_GRID {i1 in BEZ_LBLS, i2 in BEZ_SUBGRID[i1]} = GRID[0] inter interval [bez_subgrid[i1, i2, 'BEG'], bez_subgrid[i1, i2, 'END']];

# decision variables: bezier knots and control points
var bez_tau {i1 in BEZ_LBLS, i2 in BEZ_SUBGRID[i1], BEZ_GRID[i1, i2]} >= 0, <= 1;
var bez_ctls {i1 in BEZ_LBLS, BEZ_SUBGRID[i1], BEZ[i1, 'CP_ROW'], BEZ[i1, 'CP_COL']};

set BEZ_N := setof {i1 in BEZ_LBLS} bez[i1, 'CP_COL'] - 1;
set BEZ_COL {i1 in BEZ_N} := 0..i1;

param bez_B {i1 in BEZ_N, i2 in BEZ_COL[i1], i3 in BEZ_COL[i1]: i3 <= i2} = (-1)^(i2 - i3) * prod {i4 in BEZ_COL[i1]: i4 <= i2 - 1} (i1 - i4) / ((i3 - min(i4, i3 - 1)) * (i2 - i3 - min(i4, i2 - i3 - 1)));

var bez_curve {i1 in BEZ_LBLS, i2 in BEZ_SUBGRID[i1], i3 in BEZ[i1, 'DER'], i4 in BEZ[i1, 'CP_ROW'], i5 in BEZ_GRID[i1, i2]} = sum {i6 in BEZ_COL[bez[i1, 'CP_COL'] - 1], i7 in BEZ_COL[bez[i1, 'CP_COL'] - 1]: i3 <= i6 and i7 <= i6} ((prod {i8 in BEZ[i1, 'DER']: i8 < i3} (i6 - i8)) * bez_tau[i1, i2, i5]^(i6 - i3) * bez_B[bez[i1, 'CP_COL'] - 1, i6, i7] * bez_ctls[i1, i2, i4, i7]);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* bez!

# Bézier Data Section

## About the Example (WORK IN PROGRESS)
In the example below, two Bézier curves are defined in `BEZ_LBLS` with labels `0` and `1`.  As defined in `bez`, curve `0` and its 1st and 2nd derivatives will be defined in the model (`DER = 2`).  The curve is of order 2 (a quadratic curve, `CP_ROW = 2`) that will output a single value at a grid point (`CP_COL = 1`).  The curve is defined over grid points 0 to 25 as stated in `bez_subgrid` using subgrid labels defined in the set `BEZ_SUBGRID`.

> the control points for curve `0` form a $2 \times 1$ matrix of control points $\begin{bmatrix} P_0^0, & P_1^0, & P_2^0 \end{bmatrix}^T$ and Bézier curve $b^0(t)$:
>
> $b^0(t_k) = P_0^0 (1 - t_k)^2 + P_1^0 (1 - t_k) t_k + t_k^2 P_2^0$

Curve `1` is similarly defined, but it consists of a 4x6 matrix, or 6 control point vectors that are each 4 x 1.  Curves `0` and `1` each have 1 subgrid they are defined over labeled `1`.  The subgrid is defined between grid points 0 to 25 (i.e., the entire grid).

## Details
`BEZ_LBLS`: a set of lables; the labels can be arbitrary.

`bez`: a 2D table defining each curve
- `0`: a label from `BEZ_LBLS`
- `1`: a label from `BEZ_LBLS`
- `DER`: the number of derivatives to compute for the curve
- `CP_ROW`: the dimension of a control point at a grid point
- `CP_COL`: the number of control points at a grid point

`BEZ_SUBGRID`: an indexed set that is indexed by `BEZ_LBLS`.  Each indexed set is a set of sublabels that map to a consecutive set of grid points where the curve is defined, i.e., `BEZ_SUBGRID[BEZ_LBL_1] := SUBGRID_1 SUBGRID_2 ... SUBGRID_N`.  `SUBGRID_I` is defined over grid points that start at `BEG` and end at `END`.

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* bezier
set BEZ_LBLS := 0 1;
param bez := [*,:]
   :DER   CP_ROW   CP_COL :=
0   2.     2.       1.
1   2.     4.       6.;

set BEZ_SUBGRID[0] := 1;
set BEZ_SUBGRID[1] := 1;

param bez_subgrid := [*,*,:]
     :BEG   END :=
0 1   0.     25.
1 1   0.     25.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* bezier!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* state
param nq >= 0 integer;
set Q = 1..nq; # config space
set X = 1..2*nq; # state space

#----------- decision variables: generalized coordinates and forces
# state variables
var q {Q, GRID[0]};
var v {Q, GRID[0]}; # v = dq/dt
var a {Q, GRID[0]}; # a = d^2q/dt^2

# control input
var u {Q, GRID[0]};

# defined state variables
var x {i1 in X, i2 in GRID[0]} =
  if i1 <= nq then q[i1, i2]
  else v[i1-nq, i2];
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* state!

# State Data Section
## About the Example
The five-link biped walker has 4 internal joint angles and a planar floating base with coordinates $(q_x, q_y, q_\theta) \in \mathbb{R}^3$ for a total of 7 configuration variables. The example below shows how to set the number of configuration variables to 7.

## Details
`nq`: the number of configuration variables

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* state
param nq := 7.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* state!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* robot
set RBT_IDX  ordered = {'PARENT', 'S_RX', 'S_RY', 'S_RZ', 'S_PX', 'S_PY', 'S_PZ', 'Q_LB', 'Q_UB', 'F_RX', 'F_RY', 'F_RZ', 'F_PX', 'F_PY', 'F_PZ', 'F_TH', 'MASS', 'M_RX', 'M_RY', 'M_RZ', 'M_PX', 'M_PY', 'M_PZ', 'M_TH', 'IXX', 'IYY', 'IZZ', 'IXY', 'IXZ', 'IYZ'};
param robot {Q, RBT_IDX};

#----------- kinematic tree
# -1 = leaves to base, 0 = base to leaves, 1 = 1 to leaves
set SPAT_LINK_MBRS := {-1, 0, 1};

set SPAT_L {i1 in SPAT_LINK_MBRS} ordered :=
  if i1 = -1 then nq..0 by -1
  else i1..nq;

# P = parent, C = child, S = subtree, K = path to base
set SPAT_TREE_P {i1 in SPAT_L[0]} ordered = if i1 in SPAT_L[1] then {robot[i1, 'PARENT']} else {};
set SPAT_TREE_C {i1 in SPAT_L[0]} ordered = setof {i2 in SPAT_L[1], i3 in SPAT_TREE_P[i2]: i1 = i3} i2;
set SPAT_TREE_S {i1 in SPAT_L[-1]} ordered = {i1} union (union {i2 in SPAT_TREE_C[i1]} SPAT_TREE_S[i2]);
set SPAT_TREE_K {i1 in SPAT_L[0]} ordered = {i1} union (union {i2 in SPAT_TREE_P[i1]: i1 in SPAT_L[1]} SPAT_TREE_K[i2]);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* robot!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* robot
param robot := [*,:]
   :PARENT   S_RX   S_RY   S_RZ   S_PX   S_PY   S_PZ        Q_LB       Q_UB   F_RX   F_RY   F_RZ   F_PX   F_PY   F_PZ   F_TH   MASS   M_RX   M_RY   M_RZ   M_PX   M_PY   M_PZ                  M_TH                    IXX                    IYY   IZZ   IXY   IXZ   IYZ :=
1   0.        0.     0.     0.     1.     0.     0.     -Infinity   Infinity   0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.                    0.                     0.                     0.    0.    0.    0.
2   1.        0.     0.     0.     0.     0.     1.     -Infinity   Infinity   0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.                    0.                     0.                     0.    0.    0.    0.
3   2.        0.     1.     0.     0.     0.     0.     -Infinity   Infinity   0.     0.     0.     0.     0.     0.     0.     10.    0.     0.     0.     0.     0.     1.     0.1                   0.1333333333333333     0.1333333333333333     0.    0.    0.    0.
4   3.        0.     1.     0.     0.     0.     0.     -Infinity   Infinity   0.     0.     0.     0.     0.     0.     0.     4.     0.     0.     0.     0.     0.     -1.    0.08000000000000002   -0.05333333333333334   -0.05333333333333334   0.    0.    0.    0.
5   3.        0.     1.     0.     0.     0.     0.     -Infinity   Infinity   0.     0.     0.     0.     0.     0.     0.     4.     0.     0.     0.     0.     0.     -1.    0.08000000000000002   -0.05333333333333334   -0.05333333333333334   0.    0.    0.    0.
6   4.        0.     1.     0.     0.     0.     0.     -Infinity   Infinity   0.     0.     0.     0.     0.     -1.    0.4    2.     0.     0.     0.     0.     0.     -1.    0.08000000000000002   -0.02666666666666667   -0.02666666666666667   0.    0.    0.    0.
7   5.        0.     1.     0.     0.     0.     0.     -Infinity   Infinity   0.     0.     0.     0.     0.     -1.    0.4    2.     0.     0.     0.     0.     0.     -1.    0.08000000000000002   -0.02666666666666667   -0.02666666666666667   0.    0.    0.    0.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* robot!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* constraints
param nr >= 0 integer; # total number of constraints across all phases
set R = 1..nr; # constraints

set CON_IDX  ordered = {'PHASE', 'TYPE', 'ROW', 'BODY', 'R_RX', 'R_RY', 'R_RZ', 'R_PX', 'R_PY', 'R_PZ', 'F_RX', 'F_RY', 'F_RZ', 'F_PX', 'F_PY', 'F_PZ', 'F_TH', 'F_O'};
param constraint {R, CON_IDX} symbolic;

#----------- indices for physical and virtual constraints
# P = physical constraint or force, V = virtual constraint, U = virtual force
# M = constrained motions, F = constraint forces
set CON_P {i1 in PHASES} = setof {i2 in R: constraint[i2, 'PHASE'] = i1 and constraint[i2, 'TYPE'] = 'p'} constraint[i2, 'ROW'];
set CON_V {i1 in PHASES} = setof {i2 in R: constraint[i2, 'PHASE'] = i1 and constraint[i2, 'TYPE'] = 'v'} constraint[i2, 'ROW'];
set CON_U {i1 in PHASES} = setof {i2 in R: constraint[i2, 'PHASE'] = i1 and constraint[i2, 'TYPE'] = 'u'} constraint[i2, 'ROW'];

#----------- indices for physical and virtual constraints
# M = constrained motions, F = constraint forces
set CON_M {i1 in PHASES} = CON_P[i1] union CON_V[i1];
set CON_F {i1 in PHASES} = CON_P[i1] union CON_U[i1];
set CON_J {i1 in PHASES} = CON_M[i1] union CON_F[i1];

# mapping from i to k in J[p, i, j] = sum J[p, k[i], j] in phase p
set CON_R {i1 in PHASES, i2 in CON_J[i1]} := setof {i3 in R: constraint[i3, 'PHASE'] = i1 and constraint[i3, 'ROW'] = i2} i3;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* constraints!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* constraints
param nr := 20.;
param constraint := [*,:]
    :PHASE   TYPE   ROW   BODY   R_RX   R_RY   R_RZ   R_PX   R_PY   R_PZ   F_RX   F_RY   F_RZ   F_PX   F_PY   F_PZ   F_TH   F_O :=
1    1.       p      1.    6.     0.     0.     0.     1.     0.     0.     0.     0.     0.     0.     0.     -1.    0.4    0.
2    1.       p      2.    6.     0.     0.     0.     0.     0.     1.     0.     0.     0.     0.     0.     -1.    0.4    0.
3    1.       p      3.    7.     0.     0.     0.     1.     0.     0.     0.     0.     0.     0.     0.     -1.    0.4    0.
4    1.       p      4.    7.     0.     0.     0.     0.     0.     1.     0.     0.     0.     0.     0.     -1.    0.4    0.
5    1.       v      5.    4.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
6    1.       v      5.    3.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
7    1.       v      6.    5.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
8    1.       v      6.    3.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
9    1.       v      7.    6.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
10   1.       v      7.    4.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
11   1.       v      8.    7.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
12   1.       v      8.    5.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
13   1.       u      9.    4.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
14   1.       u      9.    3.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
15   1.       u      10.   5.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
16   1.       u      10.   3.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
17   1.       u      11.   6.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
18   1.       u      11.   4.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
19   1.       u      12.   7.     0.     1.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.
20   1.       u      12.   5.     0.     -1.    0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.     0.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* constraints!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* spatial transforms
param nm = 6;
set SPAT_M = 1..nm;

#----------- all transforms in robot and constraints
set SPAT_EXPC := 1..nm+1;

# constant transforms
set SPAT_XP := 1..(2 * nq + nr);

param x_jp := 1; # RBT_F
param x_im := nq + 1; # RBT_M
param x_cb := 2*nq + 1; # CON_F
  # indices into SPAT_XP

param spat_expcp {i1 in SPAT_XP, i2 in SPAT_EXPC} :=
  if x_jp <= i1 < x_im then
    robot[i1 - x_jp + 1, next('F_RX', RBT_IDX, i2 - 1)]
  else if x_im <= i1 < x_cb then
    robot[i1 - x_im + 1, next('M_RX', RBT_IDX, i2 - 1)]
  else
    constraint[i1 - x_cb + 1, next('F_RX', CON_IDX, i2 - 1)];

param Ep {i1 in SPAT_XP, i2 in SPAT_M, i3 in SPAT_M: i2 <= 3 and i3 <= 3} =
  if i2 = 1 and i3 = 1 then 1 + (1 - cos(spat_expcp[i1, 7]))*(-spat_expcp[i1, 2]^2 - spat_expcp[i1, 3]^2)
  else if i2 = 1 and i3 = 2 then (1 - cos(spat_expcp[i1, 7]))*spat_expcp[i1, 1]*spat_expcp[i1, 2] + sin(spat_expcp[i1, 7])*spat_expcp[i1, 3]
  else if i2 = 1 and i3 = 3 then -(sin(spat_expcp[i1, 7])*spat_expcp[i1, 2]) + (1 - cos(spat_expcp[i1, 7]))*spat_expcp[i1, 1]*spat_expcp[i1, 3]
  else if i2 = 2 and i3 = 1 then (1 - cos(spat_expcp[i1, 7]))*spat_expcp[i1, 1]*spat_expcp[i1, 2] - sin(spat_expcp[i1, 7])*spat_expcp[i1, 3]
  else if i2 = 2 and i3 = 2 then 1 + (1 - cos(spat_expcp[i1, 7]))*(-spat_expcp[i1, 1]^2 - spat_expcp[i1, 3]^2)
  else if i2 = 2 and i3 = 3 then sin(spat_expcp[i1, 7])*spat_expcp[i1, 1] + (1 - cos(spat_expcp[i1, 7]))*spat_expcp[i1, 2]*spat_expcp[i1, 3]
  else if i2 = 3 and i3 = 1 then sin(spat_expcp[i1, 7])*spat_expcp[i1, 2] + (1 - cos(spat_expcp[i1, 7]))*spat_expcp[i1, 1]*spat_expcp[i1, 3]
  else if i2 = 3 and i3 = 2 then -(sin(spat_expcp[i1, 7])*spat_expcp[i1, 1]) + (1 - cos(spat_expcp[i1, 7]))*spat_expcp[i1, 2]*spat_expcp[i1, 3]
  else if i2 = 3 and i3 = 3 then 1 + (1 - cos(spat_expcp[i1, 7]))*(-spat_expcp[i1, 1]^2 - spat_expcp[i1, 2]^2);

# px = (p) x = skew-symmetric matrix of p = (-E r) x
param ppx {i1 in SPAT_XP, i2 in SPAT_M, i3 in SPAT_M: i2 <= 3 and i3 <= 3} =
  if i2 = 1 and i3 = 2 then -((1 - cos(spat_expcp[i1, 7]))*(-(spat_expcp[i1, 2]*spat_expcp[i1, 4]) + spat_expcp[i1, 1]*spat_expcp[i1, 5])) + spat_expcp[i1, 6]*spat_expcp[i1, 7] - (-(spat_expcp[i1, 1]*spat_expcp[i1, 3]*spat_expcp[i1, 4]) - spat_expcp[i1, 2]*spat_expcp[i1, 3]*spat_expcp[i1, 5] - (-spat_expcp[i1, 1]^2 - spat_expcp[i1, 2]^2)*spat_expcp[i1, 6])*(-sin(spat_expcp[i1, 7]) + spat_expcp[i1, 7])
  else if i2 = 1 and i3 = 3 then (1 - cos(spat_expcp[i1, 7]))*(spat_expcp[i1, 3]*spat_expcp[i1, 4] - spat_expcp[i1, 1]*spat_expcp[i1, 6]) - spat_expcp[i1, 5]*spat_expcp[i1, 7] + (-(spat_expcp[i1, 1]*spat_expcp[i1, 2]*spat_expcp[i1, 4]) - (-spat_expcp[i1, 1]^2 - spat_expcp[i1, 3]^2)*spat_expcp[i1, 5] - spat_expcp[i1, 2]*spat_expcp[i1, 3]*spat_expcp[i1, 6])*(-sin(spat_expcp[i1, 7]) + spat_expcp[i1, 7])
  else if i2 = 2 and i3 = 1 then (1 - cos(spat_expcp[i1, 7]))*(-(spat_expcp[i1, 2]*spat_expcp[i1, 4]) + spat_expcp[i1, 1]*spat_expcp[i1, 5]) - spat_expcp[i1, 6]*spat_expcp[i1, 7] + (-(spat_expcp[i1, 1]*spat_expcp[i1, 3]*spat_expcp[i1, 4]) - spat_expcp[i1, 2]*spat_expcp[i1, 3]*spat_expcp[i1, 5] - (-spat_expcp[i1, 1]^2 - spat_expcp[i1, 2]^2)*spat_expcp[i1, 6])*(-sin(spat_expcp[i1, 7]) + spat_expcp[i1, 7])
  else if i2 = 2 and i3 = 3 then -((1 - cos(spat_expcp[i1, 7]))*(-(spat_expcp[i1, 3]*spat_expcp[i1, 5]) + spat_expcp[i1, 2]*spat_expcp[i1, 6])) + spat_expcp[i1, 4]*spat_expcp[i1, 7] - (-((-spat_expcp[i1, 2]^2 - spat_expcp[i1, 3]^2)*spat_expcp[i1, 4]) - spat_expcp[i1, 1]*spat_expcp[i1, 2]*spat_expcp[i1, 5] - spat_expcp[i1, 1]*spat_expcp[i1, 3]*spat_expcp[i1, 6])*(-sin(spat_expcp[i1, 7]) + spat_expcp[i1, 7])
  else if i2 = 3 and i3 = 1 then -((1 - cos(spat_expcp[i1, 7]))*(spat_expcp[i1, 3]*spat_expcp[i1, 4] - spat_expcp[i1, 1]*spat_expcp[i1, 6])) + spat_expcp[i1, 5]*spat_expcp[i1, 7] - (-(spat_expcp[i1, 1]*spat_expcp[i1, 2]*spat_expcp[i1, 4]) - (-spat_expcp[i1, 1]^2 - spat_expcp[i1, 3]^2)*spat_expcp[i1, 5] - spat_expcp[i1, 2]*spat_expcp[i1, 3]*spat_expcp[i1, 6])*(-sin(spat_expcp[i1, 7]) + spat_expcp[i1, 7])
  else if i2 = 3 and i3 = 2 then (1 - cos(spat_expcp[i1, 7]))*(-(spat_expcp[i1, 3]*spat_expcp[i1, 5]) + spat_expcp[i1, 2]*spat_expcp[i1, 6]) - spat_expcp[i1, 4]*spat_expcp[i1, 7] + (-((-spat_expcp[i1, 2]^2 - spat_expcp[i1, 3]^2)*spat_expcp[i1, 4]) - spat_expcp[i1, 1]*spat_expcp[i1, 2]*spat_expcp[i1, 5] - spat_expcp[i1, 1]*spat_expcp[i1, 3]*spat_expcp[i1, 6])*(-sin(spat_expcp[i1, 7]) + spat_expcp[i1, 7]);

# p x E = skew[-E.r].E = -E.skew[r].E^T.E = -E.skew[r]
param spat_Xp {i1 in SPAT_XP, i2 in SPAT_M, i3 in SPAT_M} =
  if i2 <= 3 and i3 <= 3 then Ep[i1, i2, i3]
  else if i2 > 3 and i3 > 3 then Ep[i1, i2 - 3, i3 - 3]
  else if 3 < i2 <= 6 and i3 <= 3 then 
    sum {i4 in SPAT_M: i4 <= 3} ppx[i1, i2 - 3, i4] * Ep[i1, i4, i3];

#----------- variable transforms
set SPAT_XV := 1..nq;

param x_ij := 1; # RBT_S

param spat_expcv {i1 in SPAT_XV, i2 in SPAT_M} :=
  robot[i1, next('S_RX', RBT_IDX, i2 - 1)];

var Ev {i1 in SPAT_XV, i2 in SPAT_M, i3 in SPAT_M, i4 in GRID[0]: i2 <= 3 and i3 <= 3} =
  if i2 = 1 and i3 = 1 then 1 + (1 - cos(q[i1, i4]))*(-spat_expcv[i1, 2]^2 - spat_expcv[i1, 3]^2)
  else if i2 = 1 and i3 = 2 then (1 - cos(q[i1, i4]))*spat_expcv[i1, 1]*spat_expcv[i1, 2] + sin(q[i1, i4])*spat_expcv[i1, 3]
  else if i2 = 1 and i3 = 3 then -(sin(q[i1, i4])*spat_expcv[i1, 2]) + (1 - cos(q[i1, i4]))*spat_expcv[i1, 1]*spat_expcv[i1, 3]
  else if i2 = 2 and i3 = 1 then (1 - cos(q[i1, i4]))*spat_expcv[i1, 1]*spat_expcv[i1, 2] - sin(q[i1, i4])*spat_expcv[i1, 3]
  else if i2 = 2 and i3 = 2 then 1 + (1 - cos(q[i1, i4]))*(-spat_expcv[i1, 1]^2 - spat_expcv[i1, 3]^2)
  else if i2 = 2 and i3 = 3 then sin(q[i1, i4])*spat_expcv[i1, 1] + (1 - cos(q[i1, i4]))*spat_expcv[i1, 2]*spat_expcv[i1, 3]
  else if i2 = 3 and i3 = 1 then sin(q[i1, i4])*spat_expcv[i1, 2] + (1 - cos(q[i1, i4]))*spat_expcv[i1, 1]*spat_expcv[i1, 3]
  else if i2 = 3 and i3 = 2 then -(sin(q[i1, i4])*spat_expcv[i1, 1]) + (1 - cos(q[i1, i4]))*spat_expcv[i1, 2]*spat_expcv[i1, 3]
  else if i2 = 3 and i3 = 3 then 1 + (1 - cos(q[i1, i4]))*(-spat_expcv[i1, 1]^2 - spat_expcv[i1, 2]^2);

# px = (p) x = skew-symmetric matrix of p = (-E r) x
var pvx {i1 in SPAT_XV, i2 in SPAT_M, i3 in SPAT_M, i4 in GRID[0]: i2 <= 3 and i3 <= 3} =
  if i2 = 1 and i3 = 2 then -((1 - cos(q[i1, i4]))*(-(spat_expcv[i1, 2]*spat_expcv[i1, 4]) + spat_expcv[i1, 1]*spat_expcv[i1, 5])) + spat_expcv[i1, 6]*q[i1, i4] - (-(spat_expcv[i1, 1]*spat_expcv[i1, 3]*spat_expcv[i1, 4]) - spat_expcv[i1, 2]*spat_expcv[i1, 3]*spat_expcv[i1, 5] - (-spat_expcv[i1, 1]^2 - spat_expcv[i1, 2]^2)*spat_expcv[i1, 6])*(-sin(q[i1, i4]) + q[i1, i4])
  else if i2 = 1 and i3 = 3 then (1 - cos(q[i1, i4]))*(spat_expcv[i1, 3]*spat_expcv[i1, 4] - spat_expcv[i1, 1]*spat_expcv[i1, 6]) - spat_expcv[i1, 5]*q[i1, i4] + (-(spat_expcv[i1, 1]*spat_expcv[i1, 2]*spat_expcv[i1, 4]) - (-spat_expcv[i1, 1]^2 - spat_expcv[i1, 3]^2)*spat_expcv[i1, 5] - spat_expcv[i1, 2]*spat_expcv[i1, 3]*spat_expcv[i1, 6])*(-sin(q[i1, i4]) + q[i1, i4])
  else if i2 = 2 and i3 = 1 then (1 - cos(q[i1, i4]))*(-(spat_expcv[i1, 2]*spat_expcv[i1, 4]) + spat_expcv[i1, 1]*spat_expcv[i1, 5]) - spat_expcv[i1, 6]*q[i1, i4] + (-(spat_expcv[i1, 1]*spat_expcv[i1, 3]*spat_expcv[i1, 4]) - spat_expcv[i1, 2]*spat_expcv[i1, 3]*spat_expcv[i1, 5] - (-spat_expcv[i1, 1]^2 - spat_expcv[i1, 2]^2)*spat_expcv[i1, 6])*(-sin(q[i1, i4]) + q[i1, i4])
  else if i2 = 2 and i3 = 3 then -((1 - cos(q[i1, i4]))*(-(spat_expcv[i1, 3]*spat_expcv[i1, 5]) + spat_expcv[i1, 2]*spat_expcv[i1, 6])) + spat_expcv[i1, 4]*q[i1, i4] - (-((-spat_expcv[i1, 2]^2 - spat_expcv[i1, 3]^2)*spat_expcv[i1, 4]) - spat_expcv[i1, 1]*spat_expcv[i1, 2]*spat_expcv[i1, 5] - spat_expcv[i1, 1]*spat_expcv[i1, 3]*spat_expcv[i1, 6])*(-sin(q[i1, i4]) + q[i1, i4])
  else if i2 = 3 and i3 = 1 then -((1 - cos(q[i1, i4]))*(spat_expcv[i1, 3]*spat_expcv[i1, 4] - spat_expcv[i1, 1]*spat_expcv[i1, 6])) + spat_expcv[i1, 5]*q[i1, i4] - (-(spat_expcv[i1, 1]*spat_expcv[i1, 2]*spat_expcv[i1, 4]) - (-spat_expcv[i1, 1]^2 - spat_expcv[i1, 3]^2)*spat_expcv[i1, 5] - spat_expcv[i1, 2]*spat_expcv[i1, 3]*spat_expcv[i1, 6])*(-sin(q[i1, i4]) + q[i1, i4])
  else if i2 = 3 and i3 = 2 then (1 - cos(q[i1, i4]))*(-(spat_expcv[i1, 3]*spat_expcv[i1, 5]) + spat_expcv[i1, 2]*spat_expcv[i1, 6]) - spat_expcv[i1, 4]*q[i1, i4] + (-((-spat_expcv[i1, 2]^2 - spat_expcv[i1, 3]^2)*spat_expcv[i1, 4]) - spat_expcv[i1, 1]*spat_expcv[i1, 2]*spat_expcv[i1, 5] - spat_expcv[i1, 1]*spat_expcv[i1, 3]*spat_expcv[i1, 6])*(-sin(q[i1, i4]) + q[i1, i4]);

# p x E = skew[-E.r].E = -E.skew[r].E^T.E = -E.skew[r]
var spat_Xv {i1 in SPAT_XV, i2 in SPAT_M, i3 in SPAT_M, i4 in GRID[0]} =
  if i2 <= 3 and i3 <= 3 then Ev[i1, i2, i3, i4]
  else if i2 > 3 and i3 > 3 then Ev[i1, i2 - 3, i3 - 3, i4]
  else if 3 < i2 <= 6 and i3 <= 3 then 
    sum {i5 in SPAT_M: i5 <= 3} pvx[i1, i2 - 3, i5, i4] * Ev[i1, i5, i3, i4];

#----------- track the robot's position
var spat_X_ip {i1 in SPAT_L[1], i2 in SPAT_M, i3 in SPAT_M, i4 in GRID[0]} =
  sum {i5 in SPAT_M} spat_Xv[x_ij + i1 - 1, i2, i5, i4]
    * spat_Xp[x_jp + i1 - 1, i5, i3];
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* spatial transforms!

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* M and b
#----------- used to compute compute mass matrix M in CRB and internal forces b in RNEA
param spat_ag {SPAT_M};

param spat_s_ii {i1 in SPAT_L[1], i2 in SPAT_M} =
  robot[i1, next('S_RX', RBT_IDX, i2 - 1)];

param spat_I_mm {i1 in SPAT_L[1], i2 in SPAT_M, i3 in SPAT_M} =
  if i2 = i3 and i3 > 3 then robot[i1, 'MASS']
  else if i2 = i3 and i3 <= 3 then robot[i1, next('IXX', RBT_IDX, i2 - 1)]
  else if (i2 = 1 and i3 = 2) or (i2 = 2 and i3 = 1) then robot[i1, 'IXY']
  else if (i2 = 1 and i3 = 3) or (i2 = 3 and i3 = 1) then robot[i1, 'IXZ']
  else if (i2 = 2 and i3 = 3) or (i2 = 3 and i3 = 2) then robot[i1, 'IYZ'];

param spat_I_im {i1 in SPAT_L[1], i2 in SPAT_M, i3 in SPAT_M} = sum {i4 in SPAT_M, i5 in SPAT_M} spat_Xp[x_im + i1 - 1, i4, i2] * spat_I_mm[i1, i4, i5] * spat_Xp[x_im + i1 - 1, i5, i3];
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* M and b!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* M and b
param spat_ag := [*]
1     0.
2     0.
3     0.
4     0.
5     0.
6   9.81;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* M and b!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* composite rigid body algorithm (CRB)
var CRB_IC {i1 in SPAT_L[-1], i2 in SPAT_M, i3 in SPAT_M, i4 in GRID[0] : i1 > 0} = spat_I_im[i1, i2, i3] + sum {i5 in SPAT_TREE_C[i1], i6 in SPAT_M, i7 in SPAT_M} spat_X_ip[i5, i6, i2, i4] * CRB_IC[i5, i6, i7, i4] * spat_X_ip[i5, i7, i3, i4];

var CRB_f {i1 in SPAT_L[-1], i2 in SPAT_TREE_K[i1], i3 in SPAT_M, i4 in GRID[0] : i1 > 0} = if i1 = i2 then sum {i5 in SPAT_M} CRB_IC[i1, i3, i5, i4] * spat_s_ii[i1, i5] else sum {i5 in SPAT_M} spat_X_ip[prev(i2), i5, i3, i4] * CRB_f[i1, prev(i2), i5, i4];

var M {i1 in SPAT_L[-1], i2 in SPAT_L[-1], i3 in GRID[0] : i1 > 0 && i2 > 0} = if i1 in SPAT_TREE_S[i2] then sum {i5 in SPAT_M} CRB_f[i1, i2, i5, i3] * spat_s_ii[i2, i5] else if i2 in SPAT_TREE_S[i1] then M[i2, i1, i3] else 0;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* composite rigid body algorithm (CRB)!

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* recursive Netwon-Euler algorithm (RNEA)
var RNEA_v {i1 in SPAT_L[0], i2 in SPAT_M, i3 in GRID[0]} = if i1 = 0 then 0 else spat_s_ii[i1, i2] * v[i1, i3] + sum {i4 in SPAT_TREE_P[i1], i5 in SPAT_M} spat_X_ip[i1, i2, i5, i3] * RNEA_v[i4, i5, i3];

var RNEA_vx {i1 in SPAT_L[1], i2 in SPAT_M, i3 in SPAT_M, i4 in GRID[0]} =
  if i2 = 1 and i3 = 2 then -RNEA_v[i1, 3, i4]
  else if i2 = 1 and i3 = 3 then RNEA_v[i1, 2, i4]
  else if i2 = 2 and i3 = 1 then RNEA_v[i1, 3, i4]
  else if i2 = 2 and i3 = 3 then -RNEA_v[i1, 1, i4]
  else if i2 = 3 and i3 = 1 then -RNEA_v[i1, 2, i4]
  else if i2 = 3 and i3 = 2 then RNEA_v[i1, 1, i4]
  else if i2 = 4 and i3 = 2 then -RNEA_v[i1, 6, i4]
  else if i2 = 4 and i3 = 3 then RNEA_v[i1, 5, i4]
  else if i2 = 4 and i3 = 5 then -RNEA_v[i1, 3, i4]
  else if i2 = 4 and i3 = 6 then RNEA_v[i1, 2, i4]
  else if i2 = 5 and i3 = 1 then RNEA_v[i1, 6, i4]
  else if i2 = 5 and i3 = 3 then -RNEA_v[i1, 4, i4]
  else if i2 = 5 and i3 = 4 then RNEA_v[i1, 3, i4]
  else if i2 = 5 and i3 = 6 then -RNEA_v[i1, 1, i4]
  else if i2 = 6 and i3 = 1 then -RNEA_v[i1, 5, i4]
  else if i2 = 6 and i3 = 2 then RNEA_v[i1, 4, i4]
  else if i2 = 6 and i3 = 4 then -RNEA_v[i1, 2, i4]
  else if i2 = 6 and i3 = 5 then RNEA_v[i1, 1, i4];

var RNEA_a {i1 in SPAT_L[0], i2 in SPAT_M, i3 in GRID[0]} = if i1 = 0 then spat_ag[i2] else sum {i4 in SPAT_TREE_P[i1], i5 in SPAT_M} spat_X_ip[i1, i2, i5, i3] * RNEA_a[i4, i5, i3] + (sum {i5 in SPAT_M} RNEA_vx[i1, i2, i5, i3] * spat_s_ii[i1, i5]) * v[i1, i3];

var RNEA_fb {i1 in SPAT_L[1], i2 in SPAT_M, i3 in GRID[0]} = sum {i5 in SPAT_M} spat_I_im[i1, i2, i5] * RNEA_a[i1, i5, i3] + sum {i5 in SPAT_M, i6 in SPAT_M} -RNEA_vx[i1, i5, i2, i3] * spat_I_im[i1, i5, i6] * RNEA_v[i1, i6, i3];

var RNEA_f {i1 in SPAT_L[-1], i2 in SPAT_M, i3 in GRID[0] : i1 > 0} = RNEA_fb[i1, i2, i3] + sum {i5 in SPAT_TREE_C[i1], i6 in SPAT_M} spat_X_ip[i5, i6, i2, i3] * RNEA_f[i5, i6, i3];

var b {i1 in SPAT_L[1], i3 in GRID[0]} = sum {i2 in SPAT_M} spat_s_ii[i1, i2] * RNEA_f[i1, i2, i3];
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* recursive Netwon-Euler algorithm (RNEA)!

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* task jacobian algorithm (TJA)
param OSIM_r_oo {i1 in R, i2 in SPAT_M} = constraint[i1, next('R_RX', CON_IDX, i2 - 1)];

# rigid bodies used in computation of constraints
set OSIM_K {i1 in PHASES} ordered by [0, nq] := union {i2 in CON_M[i1], i3 in CON_R[i1, i2]} SPAT_TREE_K[constraint[i3, 'BODY']];

set OSIM_B {i1 in PHASES} := setof {i2 in CON_M[i1], i3 in CON_R[i1, i2]} constraint[i3, 'BODY'];

var OSIM_R_0i {i1 in PHASES, i2 in OSIM_K[i1], i3 in SPAT_M, i4 in SPAT_M, i5 in GRID[i1]} = 
  if i2 = 0 and i3 = i4 then 1
  else sum {i6 in SPAT_TREE_P[i2], i7 in SPAT_M: (i3 <= 3 and i4 <= 3) or (i3 > 3 and i4 > 3)} OSIM_R_0i[i1, i6, i3, i7, i5] * spat_X_ip[i2, i4, i7, i5];

var OSIM_Rdot_0i {i1 in PHASES, i2 in OSIM_K[i1], i3 in SPAT_M, i4 in SPAT_M, i5 in GRID[i1]} = 
  if i2 > 0 then sum {i6 in SPAT_M: (i3 <= 3 and i4 <= 3) or (i3 > 3 and i4 > 3)} OSIM_R_0i[i1, i2, i3, i6, i5] * RNEA_vx[i2, i6, i4, i5];

var OSIM_X_bi {i1 in PHASES, i2 in OSIM_B[i1], i3 in SPAT_TREE_K[i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]} = 
  if i2 = i3 and i4 = i5 then 1 
  else if i2 > i3 then sum {i7 in SPAT_M} OSIM_X_bi[i1, i2, prev(i3), i4, i7, i6] * spat_X_ip[prev(i3), i7, i5, i6];

var OSIM_v_rel {i1 in PHASES, i2 in OSIM_B[i1], i3 in SPAT_TREE_K[i2], i4 in SPAT_M, i5 in GRID[i1]} = if i3 > 0 then (-RNEA_v[i2, i4, i5] + sum {i6 in SPAT_M} OSIM_X_bi[i1, i2, i3, i4, i6, i5] * RNEA_v[i3, i6, i5]);

var OSIM_v_relx {i1 in PHASES, i2 in OSIM_B[i1], i3 in SPAT_TREE_K[i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]} =
  if i4 = 1 and i5 = 2 then -OSIM_v_rel[i1, i2, i3, 3, i6]
  else if i4 = 1 and i5 = 3 then OSIM_v_rel[i1, i2, i3, 2, i6]
  else if i4 = 2 and i5 = 1 then OSIM_v_rel[i1, i2, i3, 3, i6]
  else if i4 = 2 and i5 = 3 then -OSIM_v_rel[i1, i2, i3, 1, i6]
  else if i4 = 3 and i5 = 1 then -OSIM_v_rel[i1, i2, i3, 2, i6]
  else if i4 = 3 and i5 = 2 then OSIM_v_rel[i1, i2, i3, 1, i6]
  else if i4 = 4 and i5 = 2 then -OSIM_v_rel[i1, i2, i3, 6, i6]
  else if i4 = 4 and i5 = 3 then OSIM_v_rel[i1, i2, i3, 5, i6]
  else if i4 = 4 and i5 = 5 then -OSIM_v_rel[i1, i2, i3, 3, i6]
  else if i4 = 4 and i5 = 6 then OSIM_v_rel[i1, i2, i3, 2, i6]
  else if i4 = 5 and i5 = 1 then OSIM_v_rel[i1, i2, i3, 6, i6]
  else if i4 = 5 and i5 = 3 then -OSIM_v_rel[i1, i2, i3, 4, i6]
  else if i4 = 5 and i5 = 4 then OSIM_v_rel[i1, i2, i3, 3, i6]
  else if i4 = 5 and i5 = 6 then -OSIM_v_rel[i1, i2, i3, 1, i6]
  else if i4 = 6 and i5 = 1 then -OSIM_v_rel[i1, i2, i3, 5, i6]
  else if i4 = 6 and i5 = 2 then OSIM_v_rel[i1, i2, i3, 4, i6]
  else if i4 = 6 and i5 = 4 then -OSIM_v_rel[i1, i2, i3, 2, i6]
  else if i4 = 6 and i5 = 5 then OSIM_v_rel[i1, i2, i3, 1, i6];

var OSIM_Xdot_bi {i1 in PHASES, i2 in OSIM_B[i1], i3 in SPAT_TREE_K[i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]} = 
  if i3 > 0 then sum {i7 in SPAT_M} OSIM_v_relx[i1, i2, i3, i4, i7, i6] * OSIM_X_bi[i1, i2, i3, i7, i5, i6];

# X_ob = T_cb = frame with axes aligned relative to the body
#       -or-
# X_ob = R_0b * R_bc * T_cb = frame with axes aligned 
# with {0} located at {c} relative to b; 0cb => (0c)b
# R_0b * R_bc * R_cb = R_0b, so apply simplification in computation of SO(3)
# R_0b * R_bc * (p x R_cb) = -R_0b * R_bc * R_cb * rx = -R_0b rx, need to compute
var OSIM_X_ob {i1 in PHASES, i2 in CON_J[i1], i3 in CON_R[i1, i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]} =
  if constraint[i3, 'F_O'] = constraint[i3, 'BODY'] then spat_Xp[x_cb + i3 - 1, i4, i5]
  else if i4 <= 3 and i5 <= 3 then OSIM_R_0i[i1, constraint[i3, 'BODY'], i4, i5, i6]
  else if i4 > 3 and i5 > 3 then OSIM_R_0i[i1, constraint[i3, 'BODY'], i4 - 3, i5 - 3, i6]
  else if i4 > 3 and i5 <= 3 then sum {i7 in SPAT_M, i8 in SPAT_M: i7 <= 3 and i8 <= 3} OSIM_R_0i[i1, constraint[i3, 'BODY'], i4 - 3, i7, i6] * Ep[x_cb + i3 - 1, i8, i7] * spat_Xp[x_cb + i3 - 1, i8 + 3, i5];

var OSIM_Xdot_ob {i1 in PHASES, i2 in CON_J[i1], i3 in CON_R[i1, i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]} =
  if constraint[i3, 'F_O'] = constraint[i3, 'BODY'] then 0
  else if i4 <= 3 and i5 <= 3 then OSIM_Rdot_0i[i1, constraint[i3, 'BODY'], i4, i5, i6]
  else if i4 > 3 and i5 > 3 then OSIM_Rdot_0i[i1, constraint[i3, 'BODY'], i4 - 3, i5 - 3, i6]
  else if i4 > 3 and i5 <= 3 then sum {i7 in SPAT_M, i8 in SPAT_M: i7 <= 3 and i8 <= 3} OSIM_Rdot_0i[i1, constraint[i3, 'BODY'], i4 - 3, i7, i6] * Ep[x_cb + i3 - 1, i8, i7] * spat_Xp[x_cb + i3 - 1, i8 + 3, i5];

# variables for computing positions and angles
# X_c0 = X_0c_0 <= this transform will give position of {c} in {0} coordinates
var OSIM_X_c0 {i1 in PHASES, i2 in CON_M[i1], i3 in CON_R[i1, i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]} = sum {i7 in SPAT_M} spat_Xp[x_cb + i3 - 1, i4, i7] * OSIM_X_bi[i1, constraint[i3, 'BODY'], 0, i7, i5, i6];

# R_0c = R_0b * R_bc <= this transform will give angles of {c} in {0} coordinates
#var OSIM_R_0c {i1 in PHASES, i2 in CON_J[i1], i3 in CON_R[i1, i2], i4 in SPAT_M, i5 in SPAT_M, i6 in GRID[i1]: i4 <= 3 and i5 <= 3} = sum {i7 in SPAT_M: i7 <= 3} OSIM_R_0i[i1, constraint[i3, 'BODY'], i4, i7, i6] * Ep[x_cb + i3 - 1, i5, i7];

var J {i1 in PHASES, i2 in CON_J[i1], i3 in SPAT_L[1], i4 in GRID[i1]} = sum {i5 in CON_R[i1, i2], i6 in SPAT_M, i7 in SPAT_M, i8 in SPAT_M : i3 in SPAT_TREE_K[constraint[i5, 'BODY']]} OSIM_r_oo[i5, i6] * OSIM_X_ob[i1, i2, i5, i6, i7, i4] * OSIM_X_bi[i1, constraint[i5, 'BODY'], i3, i7, i8, i4] * spat_s_ii[i3, i8];

var phi {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} = sum {i4 in CON_R[i1, i2], i5 in SPAT_TREE_K[constraint[i4, 'BODY']], i6 in SPAT_M, i7 in SPAT_M, i8 in SPAT_M : i5 > 0} (OSIM_r_oo[i4, i6] * (OSIM_Xdot_ob[i1, i2, i4, i6, i7, i3] * OSIM_X_bi[i1, constraint[i4, 'BODY'], i5, i7, i8, i3] + OSIM_X_ob[i1, i2, i4, i6, i7, i3] * OSIM_Xdot_bi[i1, constraint[i4, 'BODY'], i5, i7, i8, i3]) * spat_s_ii[i5, i8]) * v[i5, i3];

var OSIM_position {i1 in PHASES, i2 in CON_M[i1], i3 in CON_R[i1, i2], i4 in SPAT_M, i5 in GRID[i1]} =
  if i4 = 1 then atan2(-OSIM_X_c0[i1, i2, i3, 3, 2, i5],OSIM_X_c0[i1, i2, i3, 3, 3, i5])
  else if i4 = 2 then asin(OSIM_X_c0[i1, i2, i3, 3, 1, i5])
  else if i4 = 3 then atan2(-OSIM_X_c0[i1, i2, i3, 2, 1, i5],OSIM_X_c0[i1, i2, i3, 1, 1, i5])
  else if i4 = 4 then (-(OSIM_X_c0[i1, i2, i3, 1, 3, i5]*OSIM_X_c0[i1, i2, i3, 4, 2, i5]) + OSIM_X_c0[i1, i2, i3, 1, 2, i5]*OSIM_X_c0[i1, i2, i3, 4, 3, i5] - OSIM_X_c0[i1, i2, i3, 2, 3, i5]*OSIM_X_c0[i1, i2, i3, 5, 2, i5] + OSIM_X_c0[i1, i2, i3, 2, 2, i5]*OSIM_X_c0[i1, i2, i3, 5, 3, i5] - OSIM_X_c0[i1, i2, i3, 3, 3, i5]*OSIM_X_c0[i1, i2, i3, 6, 2, i5] + OSIM_X_c0[i1, i2, i3, 3, 2, i5]*OSIM_X_c0[i1, i2, i3, 6, 3, i5])/2
  else if i4 = 5 then (OSIM_X_c0[i1, i2, i3, 1, 3, i5]*OSIM_X_c0[i1, i2, i3, 4, 1, i5] - OSIM_X_c0[i1, i2, i3, 1, 1, i5]*OSIM_X_c0[i1, i2, i3, 4, 3, i5] + OSIM_X_c0[i1, i2, i3, 2, 3, i5]*OSIM_X_c0[i1, i2, i3, 5, 1, i5] - OSIM_X_c0[i1, i2, i3, 2, 1, i5]*OSIM_X_c0[i1, i2, i3, 5, 3, i5] + OSIM_X_c0[i1, i2, i3, 3, 3, i5]*OSIM_X_c0[i1, i2, i3, 6, 1, i5] - OSIM_X_c0[i1, i2, i3, 3, 1, i5]*OSIM_X_c0[i1, i2, i3, 6, 3, i5])/2
  else if i4 = 6 then (-(OSIM_X_c0[i1, i2, i3, 1, 2, i5]*OSIM_X_c0[i1, i2, i3, 4, 1, i5]) + OSIM_X_c0[i1, i2, i3, 1, 1, i5]*OSIM_X_c0[i1, i2, i3, 4, 2, i5] - OSIM_X_c0[i1, i2, i3, 2, 2, i5]*OSIM_X_c0[i1, i2, i3, 5, 1, i5] + OSIM_X_c0[i1, i2, i3, 2, 1, i5]*OSIM_X_c0[i1, i2, i3, 5, 2, i5] - OSIM_X_c0[i1, i2, i3, 3, 2, i5]*OSIM_X_c0[i1, i2, i3, 6, 1, i5] + OSIM_X_c0[i1, i2, i3, 3, 1, i5]*OSIM_X_c0[i1, i2, i3, 6, 2, i5])/2;

var g {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} = sum {i4 in CON_R[i1, i2], i5 in SPAT_M} OSIM_r_oo[i4, i5] * OSIM_position[i1, i2, i4, i5, i3];
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* task jacobian algorithm (TJA)!

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* eom
#----------- continuous dynamics
# decision variable: constraint forces and motion constraints
var f {i1 in PHASES, CON_F[i1], GRID[i1]};

var g_des {i1 in PHASES, CON_M[i1], GRID[i1]} default 0;
var gdot_des {i1 in PHASES, CON_M[i1], GRID[i1]} default 0;
var gddot_des {i1 in PHASES, CON_M[i1], GRID[i1]} default 0;

# constraint variables
var Jv {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} = sum {i4 in Q} J[i1,i2,i4,i3]*v[i4,i3];
var Ja {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} = sum {i4 in Q} J[i1,i2,i4,i3]*a[i4,i3];
var Jtf {i1 in PHASES, i2 in Q, i3 in GRID[i1]} = sum {i4 in CON_F[i1]} J[i1,i4,i2,i3]*f[i1,i4,i3];

# constraints
subject to CONQ {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} : g[i1,i2,i3] - g_des[i1,i2,i3] = 0;
subject to CONV {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} : Jv[i1,i2,i3] - gdot_des[i1,i2,i3] = 0;
subject to CONA {i1 in PHASES, i2 in CON_M[i1], i3 in GRID[i1]} : Ja[i1,i2,i3] + phi[i1,i2,i3] - gddot_des[i1,i2,i3] = 0;

# EL variables
var EL {i1 in Q, i2 in GRID[0]} = sum {i3 in Q} M[i1,i3,i2]*a[i3,i2] + b[i1,i2] - u[i1,i2];

# constrained equations of motion
subject to EOM {i1 in PHASES, i2 in Q, i3 in GRID[i1]} : EL[i2,i3] = Jtf[i1,i2,i3];

#----------- instantaneous impacts
set PIM dimen 2; # PIM = switching/post-impact times as pair (t-, t+)
set PIM_MBRS := {'-', '+'};
set PIM_GRID {i1 in PIM_MBRS} := setof {(i2, i3) in PIM} if i1 = '-' then i2 else i3;

# coefficient of restitution
param epsilon {i1 in PHASES, CON_M[i1], PIM_GRID['+'] inter GRID[i1]} default 0;

# decision variable: constraint impulses
var I {i1 in PHASES, CON_F[i1], PIM_GRID['+'] inter GRID[i1]};

# impact variables
var Mv {i1 in Q, (i3, i4) in PIM} = sum {i2 in Q} M[i1,i2,i3]*(v[i2,i4]-v[i2,i3]); # M(v+ - v-)

var JtI {i1 in PHASES, i2 in Q, i3 in PIM_GRID['+'] inter GRID[i1]} = sum {i4 in CON_F[i1]} J[i1,i4,i2,i3] * I[i1,i4,i3];

var eJv {i1 in PHASES, i2 in CON_M[i1], (i3, i4) in PIM: i4 in GRID[i1]} = epsilon[i1,i2,i4] * sum {i5 in Q} J[i1,i2,i5,i4]*v[i5,i3];

# pre-impact equals post-impact switching time
subject to TPI {(i1, i2) in PIM} : t[i2] = t[i1]; # t+ = t-

# plastic impact equations
subject to IME {i1 in PHASES, i2 in Q, (i3, i4) in PIM: i4 in GRID[i1]} : Mv[i2,i3,i4] - JtI[i1,i2,i4] = 0;
subject to POSTQ {i1 in PHASES, i2 in Q, (i3, i4) in PIM: i4 in GRID[i1]} : q[i2,i4] - q[i2,i3] = 0;
subject to POSTV {i1 in PHASES, i2 in CON_M[i1], (i3, i4) in PIM: i4 in GRID[i1]} : Jv[i1,i2,i4] + eJv[i1,i2,i3,i4] = 0;

# (to, from) in PIM must be unique to avoid competing constraints
check {i1 in setof {(i3,i4) in PIM} i4} : sum {i2 in setof {(i3,i4) in PIM} i4 : i2 = i1} 1 = 1;
check {i1 in setof {(i3,i4) in PIM} i3} : sum {i2 in setof {(i3,i4) in PIM} i3 : i2 = i1} 1 = 1;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* eom!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* eom
set PIM := (0, 25);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* eom!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* Runge-Kutta
set ODE_SOLVERS;
set ODE_SUBGRID {ODE_SOLVERS} dimen 2;
param ODE_K {ODE_SOLVERS};
set ODEMBRS := {0, 'BEG', 'END', 'K'};
set ODE {i1 in ODE_SOLVERS, i2 in ODEMBRS} :=
  if i2 = 0 then union {(i3, i4) in ODE_SUBGRID[i1]} i3..i4 - 1
  else if i2 = 'K' then 1..ODE_K[i1]
  else if i2 = 'BEG' then setof {(i3, i4) in ODE_SUBGRID[i1]} i3
  else if i2 = 'END' then setof {(i3, i4) in ODE_SUBGRID[i1]} i4
  else {};

var ODE_delta {i1 in SUBPHASES, i2 in ODE_SOLVERS, i3 in ODE[i2, 0] : i3 in GRID[i1]} = (t[subphases[i1, 'END']] - t[subphases[i1, 'BEG']]) / (subphases[i1, 'END'] - subphases[i1, 'BEG']);

# Butcher tableau parameters
param ODE_AK {i1 in ODE_SOLVERS, ODE[i1, 'K'], ODE[i1, 'K']};
param ODE_bK {i1 in ODE_SOLVERS, ODE[i1, 'K']};
param ODE_cK {i1 in ODE_SOLVERS, ODE[i1, 'K']};

# decision variable: step size
var ODE_hK {i1 in ODE_SOLVERS, ODE[i1, 0]} >= 0; # time step size

param ODE_a0 {i1 in ODE_SOLVERS} = if 0 = sum {i2 in ODE[i1, 'K']} ODE_AK[i1, 1, i2] then 1;
param ODE_aK {i1 in ODE_SOLVERS} = if 0 = sum {i2 in ODE[i1, 'K']} (ODE_AK[i1, ODE_K[i1], i2] - ODE_bK[i1, i2]) then 1;

set ODE_GRID ordered by [0, GRID_MAX] := union {i2 in ODE_SOLVERS} 
  (setof {i4 in ODE[i2, 0], i5 in ODE[i2, 'K']: 
    1 + ODE_a0[i2] <= i5 <= ODE_K[i2] - ODE_aK[i2]} i4 + i5/10^ceil(log10(ODE_K[i2]+1)) 
  union ODE[i2, 0] union ODE[i2, 'END']);

# q defect constraints
subject to ODE_DEFQ {i1 in ODE_SOLVERS, i2 in Q, i3 in ODE[i1, 0]} : q[i2,next(i3,ODE_GRID,ODE_K[i1]+1-ODE_a0[i1]-ODE_aK[i1])] = q[i2,i3] + ODE_hK[i1,i3] * sum {i4 in ODE[i1,'K']} ODE_bK[i1,i4] * v[i2,next(i3,ODE_GRID,i4-ODE_a0[i1])];

# q collocation constraints
subject to ODE_COLQ {i1 in ODE_SOLVERS, i2 in Q, i3 in ODE[i1, 0], i4 in ODE[i1, 'K']: 1 + ODE_a0[i1] <= i4 <= ODE_K[i1] - ODE_aK[i1]} : q[i2, next(i3,ODE_GRID,i4-ODE_a0[i1])] = q[i2,i3] + ODE_hK[i1,i3] * sum {i5 in ODE[i1, 'K']} ODE_AK[i1,i4,i5] * v[i2,next(i3,ODE_GRID,i5-ODE_a0[i1])];

# v defect constraints
subject to ODE_DEFV {i1 in ODE_SOLVERS, i2 in Q, i3 in ODE[i1, 0]} : v[i2,next(i3,ODE_GRID,ODE_K[i1]+1-ODE_a0[i1]-ODE_aK[i1])] = v[i2,i3] + ODE_hK[i1,i3] * sum {i4 in ODE[i1,'K']} ODE_bK[i1,i4] * a[i2,next(i3,ODE_GRID,i4-ODE_a0[i1])];

# v collocation constraints
subject to ODE_COLV {i1 in ODE_SOLVERS, i2 in Q, i3 in ODE[i1, 0], i4 in ODE[i1, 'K']: 1 + ODE_a0[i1] <= i4 <= ODE_K[i1] - ODE_aK[i1]} : v[i2, next(i3,ODE_GRID,i4-ODE_a0[i1])] = v[i2,i3] + ODE_hK[i1,i3] * sum {i5 in ODE[i1, 'K']} ODE_AK[i1,i4,i5] * a[i2,next(i3,ODE_GRID,i5-ODE_a0[i1])];

# time defects constraints
subject to ODE_DEFTIME {i1 in ODE_SOLVERS, i2 in ODE[i1, 0]} : t[next(i2, ODE_GRID, ODE_K[i1]+1-ODE_a0[i1]-ODE_aK[i1])] = t[i2] + ODE_hK[i1, i2];

# time collocation constraints
subject to ODE_COLTIME {i1 in ODE_SOLVERS, i2 in ODE[i1, 0], i3 in ODE[i1, 'K']: 1 + ODE_a0[i1] <= i3 <= ODE_K[i1] - ODE_aK[i1]} : t[next(i2, ODE_GRID, i3-ODE_a0[i1])] = t[i2] + ODE_hK[i1, i2] * ODE_cK[i1, i3];

# step size constraints
subject to ODE_DEFSTEP {i1 in SUBPHASES, i2 in ODE_SOLVERS, i3 in ODE[i2, 0]: i3 in GRID[i1]} : ODE_hK[i2, i3] = (next(i3, ODE_GRID, ODE_K[i2]+1-ODE_a0[i2]-ODE_aK[i2]) - i3) * ODE_delta[i1, i2, i3];

# knots points must be integers
check {i1 in ODE_SOLVERS, (i2, i3) in ODE_SUBGRID[i1]} : i2 in Integers and i3 in Integers;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* Runge-Kutta!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* Runge-Kutta
set ODE_SOLVERS := HERM_SIMP;
param ODE_K := [*]
HERM_SIMP   3.;
param ODE_AK := 
[HERM_SIMP,*,:]
                   :1                    2                   3 :=
1   0.                   0.                   0.
2   0.2083333333333333   0.3333333333333333   -0.04166666666666666
3   0.1666666666666667   0.6666666666666666   0.1666666666666667;
param ODE_bK := [*,:]
                           :1                    2                 3 :=
HERM_SIMP   0.1666666666666667   0.6666666666666666   0.1666666666666667;
param ODE_cK := [*,:]
           :1     2   3 :=
HERM_SIMP   0.   0.5   1.;
set ODE_SUBGRID[HERM_SIMP] := (0, 25);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* Runge-Kutta!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# model has been processed, so now add data specific rules for modules in the model file
redeclare set MOD_GRID {i1 in MOD_MODULES} := 
  if i1 = 'ODE' then ODE_GRID
  else if i1 = 'PIM' then union {(i2,i3) in PIM} {i2, i3}
  else if i1 = 'BEZ' then union {i2 in BEZ_LBLS, i3 in BEZ_SUBGRID[i2]} bez_subgrid[i2, i3, 'BEG']..bez_subgrid[i2, i3, 'END']
  else {};
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid
# activate model-define modules in the model file
set MOD_MODULES := ODE PIM BEZ;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* grid!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* vhc
# this module assumes 
#  1) indices used for g_des and friends can be reused for 
#     indexing into bez_curve and friends,
#  2) only one phase with value 1, and
#  3) GRID[1] = BEZ_GRID[1, 1]

#-- virtual holonomic constraints
param vhc_c {Q};
param vhc_Kp {CON_V[1], CON_V[1]};
param vhc_Kv {CON_V[1], CON_V[1]};
  # Kp and Kv are premultiplied by 1/epsilon^2 and 1/epsilon

# clock variable (state-based w.r.t. x hip position)
var vhc_tau0 = sum {i2 in Q} vhc_c[i2] * q[i2, 0];
var vhc_tauf = sum {i2 in Q} vhc_c[i2] * q[i2, GRID_MAX];
var vhc_taudot {i1 in GRID[1]} = (sum {i2 in Q} vhc_c[i2] * v[i2, i1])/(vhc_tauf - vhc_tau0);
var vhc_tauddot {i1 in GRID[1]} = (sum {i2 in Q} vhc_c[i2] * a[i2, i1])/(vhc_tauf - vhc_tau0);

var vhc_v {i2 in CON_V[1], i3 in GRID[1]} = sum {i4 in CON_V[1]} (vhc_Kp[i2, i4] * (g[1, i4, i3] - g_des[1, i4, i3]) + vhc_Kv[i2, i4] * (Jv[1, i4, i3] - gdot_des[1, i4, i3]));

subject to VHC_CLCK {i1 in GRID[1]} : bez_tau[1, 1, i1] = ((sum {i2 in Q} vhc_c[i2] * q[i2, i1]) - vhc_tau0)/(vhc_tauf - vhc_tau0);

subject to VHC_Q {i2 in CON_V[1], i3 in GRID[1]} : g_des[1, i2, i3] = bez_curve[1, 1, 0, i2 - 5, i3];

subject to VHC_V {i2 in CON_V[1], i3 in GRID[1]} : gdot_des[1, i2, i3] = bez_curve[1, 1, 1, i2 - 5, i3] * vhc_taudot[i3];

subject to VHC_A {i2 in CON_V[1], i3 in GRID[1]} : gddot_des[1, i2, i3] = bez_curve[1, 1, 2, i2 - 5, i3] * vhc_taudot[i3]^2 + bez_curve[1, 1, 1, i2 - 5, i3] * vhc_tauddot[i3] - vhc_v[i2, i3];

#-- physical holonomic constraints
param phc_Kp {CON_P[1], CON_P[1]};
param phc_Kv {CON_P[1], CON_P[1]};

# clock variable (time-based w.r.t. gait period)
var phc_tau0 = t[0];
var phc_tauf = t[GRID_MAX];
var phc_taudot {i1 in GRID[1]} = 1 / (phc_tauf - phc_tau0);
param phc_tauddot {i1 in GRID[1]} = 0;

var phc_v {i2 in CON_P[1], i3 in GRID[1]: i2 <= 2} = sum {i4 in CON_P[1]: i2 <= 2} (phc_Kp[i2, i4] * (g[1, i4, i3] - g_des[1, i4, i3]) + phc_Kv[i2, i4] * (Jv[1, i4, i3] - gdot_des[1, i4, i3]));

subject to PHC_CLCK {i1 in GRID[1]} : bez_tau[0, 1, i1] = (t[i1] - phc_tau0)/(phc_tauf - phc_tau0);

subject to PHC_Q {i2 in CON_P[1], i3 in GRID[1]: i2 <= 2} : g_des[1, i2, i3] = bez_curve[0, 1, 0, i2 - 1, i3];

subject to PHC_V {i2 in CON_P[1], i3 in GRID[1]: i2 <= 2} : gdot_des[1, i2, i3] = bez_curve[0, 1, 1, i2 - 1, i3] * phc_taudot[i3];

subject to PHC_A {i2 in CON_P[1], i3 in GRID[1]: i2 <= 2} : gddot_des[1, i2, i3] = bez_curve[0, 1, 2, i2 - 1, i3] * phc_taudot[i3]^2 - phc_v[i2, i3];
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* vhc!

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* gait
# constraints are set up so that
# 1-2 => left leg (x, z)
# 3-4 => right leg (x, z) [swing leg for grid points 0..GRID_MAX]
# 5-8 => vhc q[4, *]-q[7,*]
# 9-12 => vhc transmission matrix B[4, *]-B[7, *]

set GAIT_COORDS;

param mu;
param step_length_des >= 0;
param step_time_des >= 0;
param GAIT_A {GAIT_COORDS, GAIT_COORDS};

#-- step period
subject to T0: t[0] = 0;
subject to TF: 0.25 <= t[GRID_MAX] <= 0.75;

#-- step length
subject to STEP_LENGTH : 0.1 <= g[1, 3, GRID_MAX] <= 0.5;

#-- step height
subject to STEP_HEIGHT : g[1, 4, GRID_MAX] = 0;

#-- walking speed
subject to WALKING_SPEED : g[1, 3, GRID_MAX] = step_length_des / step_time_des * t[GRID_MAX];

#-- theta dot
subject to THETA_DOT {i1 in GRID[1]} : 0.05 <= vhc_taudot[i1];

#-- swing foot height
subject to SWING_FOOT_HEIGHT : 0.1 <= g[1, 4, member(ceil(card(GRID[1]) / 2), GRID[1])];

#-- ground clearance
subject to GROUND_CLEARANCE {i1 in GRID[1]} : g[1, 4, i1] >= 0;

#-- stance position at 0
subject to STANCE_POSITION_AT_0 {i1 in CON_P[1]: i1 <= 2} : g[1, i1, 0] = 0;

#-- stance velocity at 0
subject to STANCE_VELOCITY_AT_0 {i1 in CON_P[1]: i1 <= 2} : Jv[1, i1, 0] = 0;

#-- swing-leg impact velocity
subject to SWING_IMPACT_VELX : -0.01 <= Jv[1, 3, GRID_MAX] <= 0.01;
subject to SWING_IMPACT_VELZ : Jv[1, 4, GRID_MAX] <= -0.25;

#-- bezier coefficients
subject to BEZIER_COEFFS {i1 in BEZ_LBLS, i2 in BEZ[i1, 'CP_ROW'], i3 in BEZ[i1, 'CP_COL']} : -5 <= bez_ctls[i1, 1, i2, i3] <= 5;

#-- periodicity constraints
subject to PERIODICITY_CONSTRAINT_Q {i1 in Q: i1 > 1} : q[i1, 0] - sum {i2 in Q: i1 > 1} GAIT_A[i1, i2] * q[i2, GRID_MAX] = 0;

var M0 {i1 in Q, i2 in Q} = M[i1, i2, 0];
var Rvf {i1 in Q} = sum {i2 in Q} GAIT_A[i1, i2] * v[i2, GRID_MAX];
var J0I {i1 in Q} = sum {i4 in CON_P[1]: i4 <= 2} J[1, i4, i1, 0] * I[1, i4, GRID_MAX];
var MOM0 {i1 in Q} = sum {i2 in Q} M0[i1, i2] * (v[i2, 0] - Rvf[i2]);
subject to PERIODICITY_CONSTRAINT_IME {i1 in Q} : MOM0[i1] - J0I[i1] = 0;

#-- friction
subject to PYRAMID_POINT_PLANAR_LB {i1 in GRID[1]} : -mu * f[1, 2, i1] <= f[1, 1, i1];

subject to PYRAMID_POINT_PLANAR_UB {i1 in GRID[1]} : f[1, 1, i1] <= mu * f[1, 2, i1];

subject to NORMAL_FORCE {i1 in GRID[1]} : f[1, 2, i1] >= 0;

#-- state and input constraints
subject to V_BNDS {i1 in Q, i2 in GRID[1]} : -20 <= v[i1, i2] <= 20;
subject to A_BNDS {i1 in Q, i2 in GRID[1]} : -100 <= a[i1, i2] <= 100;

subject to FC_BNDS {i1 in CON_P[1], i2 in GRID[1]: i1 <= 2} : -1000 <= f[1, i1, i2] <= 1000;
subject to F_ZERO {i1 in CON_P[1], i2 in GRID[1]: 2 < i1} : f[1, i1, i2] = 0;

subject to U_BNDS {i1 in CON_U[1], i2 in GRID[1]} : -100 <= f[1, i1, i2] <= 100;
subject to U_ZERO {i1 in Q, i2 in GRID[1]} : u[i1, i2] = 0;

subject to TORSO_PX {i1 in GRID[1]} : -0.5 <= q[1, i1] <= 0.5;
subject to TORSO_PZ {i1 in GRID[1]} : 0.5 <= q[2, i1] <= 1.25;
subject to TORSO_RY {i1 in GRID[1]} : -acos(sqrt(3)/2) <= q[3, i1] <= acos(sqrt(3)/2);

subject to HIPS {i1 in Q, i2 in GRID[1]: 4 <= i1 <= 5} : -acos(-1) <= q[i1, i2] <= acos(-1);
subject to KNEES {i1 in Q, i2 in GRID[1]: i1 >= 6} : 5 * acos(-1) / 180 <= q[i1, i2] <= acos(-1);
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* gait!

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* five-link tropic
param vhc_c := [*]
1   1.
2   0.
3   0.
4   0.
5   0.
6   0.
7   0.;
set GAIT_COORDS := 1 2 3 4 5 6 7 8 9 10 11 12 13 14;
param GAIT_A := [*,:]
    :1    2    3    4    5    6    7    8    9   10   11   12   13   14 :=
1    1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
2    0.   1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
3    0.   0.   1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
4    0.   0.   0.   0.   1.   0.   0.   0.   0.   0.   0.   0.   0.   0.
5    0.   0.   0.   1.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
6    0.   0.   0.   0.   0.   0.   1.   0.   0.   0.   0.   0.   0.   0.
7    0.   0.   0.   0.   0.   1.   0.   0.   0.   0.   0.   0.   0.   0.
8    0.   0.   0.   0.   0.   0.   0.   1.   0.   0.   0.   0.   0.   0.
9    0.   0.   0.   0.   0.   0.   0.   0.   1.   0.   0.   0.   0.   0.
10   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.   0.   0.   0.   0.
11   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.   0.   0.
12   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.   0.   0.   0.
13   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.
14   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   1.   0.;
param mu := 0.6;
param step_length_des := 0.4;
param step_time_des := 1.;
param vhc_Kp := [*,:]
     :5      6      7   8 :=
5   100.   0.     0.     0.
6   0.     100.   0.     0.
7   0.     0.     100.   0.
8   0.     0.     0.     100.;
param vhc_Kv := [*,:]
    :5     6     7   8 :=
5   20.   0.    0.    0.
6   0.    20.   0.    0.
7   0.    0.    20.   0.
8   0.    0.    0.    20.;
param phc_Kp := [*,:]
     :1      2      3   4 :=
1   100.   0.     0.     0.
2   0.     100.   0.     0.
3   0.     0.     100.   0.
4   0.     0.     0.     100.;
param phc_Kv := [*,:]
    :1     2     3   4 :=
1   20.   0.    0.    0.
2   0.    20.   0.    0.
3   0.    0.    20.   0.
4   0.    0.    0.    20.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* five-link tropic!
model;

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* objective
var uhzd = sum {i1 in ODE_SOLVERS, i2 in ODE[i1, 0]} ODE_hK[i1, i2] * sum {i3 in ODE[i1, 'K']} (ODE_bK[i1, i3] * (sum {i4 in PHASES, i5 in CON_U[i4]: i2 in GRID[i4]} f[i4, i5, next(i2,ODE_GRID,i3-ODE_a0[i1])]^2));

minimize UHZD : uhzd;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* objective!

# The Initial Guess
The initial guess is stored in an AMPL data file along with other data-relevant statements.  For the trajectory being optimized, initial guesses are provided for the
* configuration q,
* velocity v,
* acceleration a,
* time t,
* phyiscal and virtual forces f,
* impulse I, and
* Bézier control points, bez_ctls

In [ ]:
%%ampl_eval
data;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* seed
var q := [:,*]
                       :1       2                      3                      4                      5                      6                   7 :=
0      0.                    1.      0.                     0.                     0.                     0.                     0.
0.2    0.004                 1.004   0.004                  0.004                  0.004                  0.004                  0.004
1      0.008                 1.008   0.008                  0.008                  0.008                  0.008                  0.008
1.2    0.012                 1.012   0.012                  0.012                  0.012                  0.012                  0.012
2      0.016                 1.016   0.016                  0.016                  0.016                  0.016                  0.016
2.2    0.02                  1.02    0.02                   0.02                   0.02                   0.02                   0.02
3      0.024                 1.024   0.024                  0.024                  0.024                  0.024                  0.024
3.2    0.028                 1.028   0.028                  0.028                  0.028                  0.028                  0.028
4      0.032                 1.032   0.032                  0.032                  0.032                  0.032                  0.032
4.2    0.036                 1.036   0.036                  0.036                  0.036                  0.036                  0.036
5      0.04000000000000001   1.04    0.04000000000000001    0.04000000000000001    0.04000000000000001    0.04000000000000001    0.04000000000000001
5.2    0.044                 1.044   0.044                  0.044                  0.044                  0.044                  0.044
6      0.048                 1.048   0.048                  0.048                  0.048                  0.048                  0.048
6.2    0.052                 1.052   0.052                  0.052                  0.052                  0.052                  0.052
7      0.056                 1.056   0.056                  0.056                  0.056                  0.056                  0.056
7.2    0.06                  1.06    0.06                   0.06                   0.06                   0.06                   0.06
8      0.064                 1.064   0.064                  0.064                  0.064                  0.064                  0.064
8.2    0.06800000000000001   1.068   0.06800000000000001    0.06800000000000001    0.06800000000000001    0.06800000000000001    0.06800000000000001
9      0.07199999999999999   1.072   0.07199999999999999    0.07199999999999999    0.07199999999999999    0.07199999999999999    0.07199999999999999
9.2    0.07600000000000001   1.076   0.07600000000000001    0.07600000000000001    0.07600000000000001    0.07600000000000001    0.07600000000000001
10     0.08000000000000002   1.08    0.08000000000000002    0.08000000000000002    0.08000000000000002    0.08000000000000002    0.08000000000000002
10.2   0.084                 1.084   0.084                  0.084                  0.084                  0.084                  0.084
11     0.088                 1.088   0.088                  0.088                  0.088                  0.088                  0.088
11.2   0.092                 1.092   0.092                  0.092                  0.092                  0.092                  0.092
12     0.096                 1.096   0.096                  0.096                  0.096                  0.096                  0.096
12.2   0.1                   1.1     0.1                    0.1                    0.1                    0.1                    0.1
13     0.104                 1.096   0.096                  0.096                  0.096                  0.096                  0.096
13.2   0.108                 1.092   0.092                  0.092                  0.092                  0.092                  0.092
14     0.112                 1.088   0.088                  0.088                  0.088                  0.088                  0.088
14.2   0.116                 1.084   0.084                  0.084                  0.084                  0.084                  0.084
15     0.12                  1.08    0.08000000000000002    0.08000000000000002    0.08000000000000002    0.08000000000000002    0.08000000000000002
15.2   0.124                 1.076   0.07600000000000001    0.07600000000000001    0.07600000000000001    0.07600000000000001    0.07600000000000001
16     0.128                 1.072   0.07200000000000001    0.07200000000000001    0.07200000000000001    0.07200000000000001    0.07200000000000001
16.2   0.132                 1.068   0.06800000000000001    0.06800000000000001    0.06800000000000001    0.06800000000000001    0.06800000000000001
17     0.136                 1.064   0.064                  0.064                  0.064                  0.064                  0.064
17.2   0.14                  1.06    0.05999999999999998    0.05999999999999998    0.05999999999999998    0.05999999999999998    0.05999999999999998
18     0.144                 1.056   0.056                  0.056                  0.056                  0.056                  0.056
18.2   0.148                 1.052   0.052                  0.052                  0.052                  0.052                  0.052
19     0.152                 1.048   0.048                  0.048                  0.048                  0.048                  0.048
19.2   0.156                 1.044   0.044                  0.044                  0.044                  0.044                  0.044
20     0.16                  1.04    0.03999999999999999    0.03999999999999999    0.03999999999999999    0.03999999999999999    0.03999999999999999
20.2   0.164                 1.036   0.03599999999999999    0.03599999999999999    0.03599999999999999    0.03599999999999999    0.03599999999999999
21     0.168                 1.032   0.03200000000000002    0.03200000000000002    0.03200000000000002    0.03200000000000002    0.03200000000000002
21.2   0.172                 1.028   0.02800000000000002    0.02800000000000002    0.02800000000000002    0.02800000000000002    0.02800000000000002
22     0.176                 1.024   0.024                  0.024                  0.024                  0.024                  0.024
22.2   0.18                  1.02    0.01999999999999999    0.01999999999999999    0.01999999999999999    0.01999999999999999    0.01999999999999999
23     0.184                 1.016   0.01599999999999999    0.01599999999999999    0.01599999999999999    0.01599999999999999    0.01599999999999999
23.2   0.188                 1.012   0.01199999999999998    0.01199999999999998    0.01199999999999998    0.01199999999999998    0.01199999999999998
24     0.192                 1.008   0.008000000000000007   0.008000000000000007   0.008000000000000007   0.008000000000000007   0.008000000000000007
24.2   0.196                 1.004   0.004000000000000004   0.004000000000000004   0.004000000000000004   0.004000000000000004   0.004000000000000004
25     0.2                   1.      0.                     0.                     0.                     0.                     0.;
var v := [:,*]
       :1                     2      3      4      5      6   7 :=
0      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
0.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
1      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
1.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
2      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
2.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
3      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
3.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
4      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
4.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
5      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
5.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
6      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
6.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
7      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
7.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
8      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
8.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
9      0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
9.2    0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
10     0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
10.2   0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
11     0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
11.2   0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
12     0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
12.2   0.4   0.4000000000000004    0.4    0.4    0.4    0.4    0.4
13     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
13.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
14     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
14.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
15     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
15.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
16     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
16.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
17     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
17.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
18     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
18.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
19     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
19.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
20     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
20.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
21     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
21.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
22     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
22.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
23     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
23.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
24     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
24.2   0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4
25     0.4   -0.4000000000000004   -0.4   -0.4   -0.4   -0.4   -0.4;
var a := [:,*]
      :1      2                    3                    4                    5                    6                 7 :=
0      0.   0.     0.                   0.                   0.                   0.                   0.
0.2    0.   0.     0.                   0.                   0.                   0.                   0.
1      0.   0.     0.                   0.                   0.                   0.                   0.
1.2    0.   0.     0.                   0.                   0.                   0.                   0.
2      0.   0.     0.                   0.                   0.                   0.                   0.
2.2    0.   0.     0.                   0.                   0.                   0.                   0.
3      0.   0.     0.                   0.                   0.                   0.                   0.
3.2    0.   0.     0.                   0.                   0.                   0.                   0.
4      0.   0.     0.                   0.                   0.                   0.                   0.
4.2    0.   0.     0.                   0.                   0.                   0.                   0.
5      0.   0.     0.                   0.                   0.                   0.                   0.
5.2    0.   0.     0.                   0.                   0.                   0.                   0.
6      0.   0.     0.                   0.                   0.                   0.                   0.
6.2    0.   0.     0.                   0.                   0.                   0.                   0.
7      0.   0.     0.                   0.                   0.                   0.                   0.
7.2    0.   0.     0.                   0.                   0.                   0.                   0.
8      0.   0.     0.                   0.                   0.                   0.                   0.
8.2    0.   0.     0.                   0.                   0.                   0.                   0.
9      0.   0.     0.                   0.                   0.                   0.                   0.
9.2    0.   0.     0.                   0.                   0.                   0.                   0.
10     0.   0.     0.                   0.                   0.                   0.                   0.
10.2   0.   0.     0.                   0.                   0.                   0.                   0.
11     0.   0.     0.                   0.                   0.                   0.                   0.
11.2   0.   0.     0.                   0.                   0.                   0.                   0.
12     0.   0.     0.                   0.                   0.                   0.                   0.
12.2   0.   -40.   -39.99999999999997   -39.99999999999997   -39.99999999999997   -39.99999999999997   -39.99999999999997
13     0.   -40.   -39.99999999999997   -39.99999999999997   -39.99999999999997   -39.99999999999997   -39.99999999999997
13.2   0.   0.     0.                   0.                   0.                   0.                   0.
14     0.   0.     0.                   0.                   0.                   0.                   0.
14.2   0.   0.     0.                   0.                   0.                   0.                   0.
15     0.   0.     0.                   0.                   0.                   0.                   0.
15.2   0.   0.     0.                   0.                   0.                   0.                   0.
16     0.   0.     0.                   0.                   0.                   0.                   0.
16.2   0.   0.     0.                   0.                   0.                   0.                   0.
17     0.   0.     0.                   0.                   0.                   0.                   0.
17.2   0.   0.     0.                   0.                   0.                   0.                   0.
18     0.   0.     0.                   0.                   0.                   0.                   0.
18.2   0.   0.     0.                   0.                   0.                   0.                   0.
19     0.   0.     0.                   0.                   0.                   0.                   0.
19.2   0.   0.     0.                   0.                   0.                   0.                   0.
20     0.   0.     0.                   0.                   0.                   0.                   0.
20.2   0.   0.     0.                   0.                   0.                   0.                   0.
21     0.   0.     0.                   0.                   0.                   0.                   0.
21.2   0.   0.     0.                   0.                   0.                   0.                   0.
22     0.   0.     0.                   0.                   0.                   0.                   0.
22.2   0.   0.     0.                   0.                   0.                   0.                   0.
23     0.   0.     0.                   0.                   0.                   0.                   0.
23.2   0.   0.     0.                   0.                   0.                   0.                   0.
24     0.   0.     0.                   0.                   0.                   0.                   0.
24.2   0.   0.     0.                   0.                   0.                   0.                   0.
25     0.   0.     0.                   0.                   0.                   0.                   0.;
var t := [*]
0                       0.
0.2                   0.01
1                     0.02
1.2                   0.03
2                     0.04
2.2                   0.05
3                     0.06
3.2    0.07000000000000001
4                     0.08
4.2                   0.09
5                      0.1
5.2                   0.11
6                     0.12
6.2                   0.13
7                     0.14
7.2                   0.15
8                     0.16
8.2                   0.17
9                     0.18
9.2                   0.19
10                     0.2
10.2                  0.21
11                    0.22
11.2                  0.23
12                    0.24
12.2                  0.25
13                    0.26
13.2                  0.27
14                    0.28
14.2                  0.29
15                     0.3
15.2                  0.31
16                    0.32
16.2                  0.33
17                    0.34
17.2                  0.35
18                    0.36
18.2                  0.37
19                    0.38
19.2                  0.39
20                     0.4
20.2                  0.41
21                    0.42
21.2                  0.43
22                    0.44
22.2                  0.45
23                    0.46
23.2                  0.47
24                    0.48
24.2                  0.49
25                     0.5;
var f := [*,:,*]
        :1    2    3    4                                    9                                   10                                    11                                 12 :=
1 0      0.   0.   0.   0.   0.00000000000000001110223024625156   0.00000000000000001110223024625156   -0.00000000000000001387778780781446   -0.00000000000000001387778780781446
1 0.2    0.   0.   0.   0.   0.1065266109331879                   0.1065266109331879                   0.0189985875215484                    0.0189985875215484
1 1      0.   0.   0.   0.   0.2130448876156095                   0.2130448876156095                   0.03799446025050756                   0.03799446025050756
1 1.2    0.   0.   0.   0.   0.3195464965470149                   0.3195464965470149                   0.0569849037848781                    0.0569849037848781
1 2      0.   0.   0.   0.   0.426023105728108                    0.426023105728108                    0.07596720450378436                   0.07596720450378436
1 2.2    0.   0.   0.   0.   0.5324663854108245                   0.5324663854108245                   0.0949386499578952                    0.0949386499578952
1 3      0.   0.   0.   0.   0.6388680088483736                   0.6388680088483736                   0.113896529259677                     0.113896529259677
1 3.2    0.   0.   0.   0.   0.7452196530449601                   0.7452196530449601                   0.1328381334734208                    0.1328381334734208
1 4      0.   0.   0.   0.   0.851512999505115                    0.851512999505115                    0.1517607560049893                    0.1517607560049893
1 4.2    0.   0.   0.   0.   0.957739734982543                    0.957739734982543                    0.1706616929912262                    0.1706616929912262
1 5      0.   0.   0.   0.   1.063891552228427                    1.063891552228427                    0.1895382436889724                    0.1895382436889724
1 5.2    0.   0.   0.   0.   1.169960150739085                    1.169960150739085                    0.2083877108636329                    0.2083877108636329
1 6      0.   0.   0.   0.   1.275937237502928                    1.275937237502928                    0.2272074011772387                    0.2272074011772387
1 6.2    0.   0.   0.   0.   1.381814527746617                    1.381814527746617                    0.2459946255759476                    0.2459946255759476
1 7      0.   0.   0.   0.   1.487583745680359                    1.487583745680359                    0.2647466996769285                    0.2647466996769285
1 7.2    0.   0.   0.   0.   1.593236625242241                    1.593236625242241                    0.2834609441545717                    0.2834609441545717
1 8      0.   0.   0.   0.   1.698764910841553                    1.698764910841553                    0.3021346851259734                    0.3021346851259734
1 8.2    0.   0.   0.   0.   1.804160358100995                    1.804160358100995                    0.3207652545356344                    0.3207652545356344
1 9      0.   0.   0.   0.   1.909414734597699                    1.909414734597699                    0.3393499905393206                    0.3393499905393206
1 9.2    0.   0.   0.   0.   2.014519820603001                    2.014519820603001                    0.35788623788703                      0.35788623788703
1 10     0.   0.   0.   0.   2.119467409820853                    2.119467409820853                    0.3763713483050079                    0.3763713483050079
1 10.2   0.   0.   0.   0.   2.224249310124842                    2.224249310124842                    0.3948026808767609                    0.3948026808767609
1 11     0.   0.   0.   0.   2.328857344293692                    2.328857344293692                    0.4131776024230083                    0.4131776024230083
1 11.2   0.   0.   0.   0.   2.433283350745206                    2.433283350745206                    0.4314934878805216                    0.4314934878805216
1 12     0.   0.   0.   0.   2.53751918426855                     2.53751918426855                     0.4497477206797952                    0.4497477206797952
1 12.2   0.   0.   0.   0.   -42.60354527438855                   -42.60354527438855                   -4.853812955734573                    -4.853812955734573
1 13     0.   0.   0.   0.   -42.28755119469271                   -42.28755119469271                   -4.800502500202259                    -4.800502500202259
1 13.2   0.   0.   0.   0.   2.433283350745206                    2.433283350745206                    0.4314934878805216                    0.4314934878805216
1 14     0.   0.   0.   0.   2.328857344293692                    2.328857344293692                    0.4131776024230081                    0.4131776024230081
1 14.2   0.   0.   0.   0.   2.224249310124843                    2.224249310124843                    0.3948026808767609                    0.3948026808767609
1 15     0.   0.   0.   0.   2.119467409820853                    2.119467409820853                    0.376371348305008                     0.376371348305008
1 15.2   0.   0.   0.   0.   2.014519820603                       2.014519820603                       0.35788623788703                      0.35788623788703
1 16     0.   0.   0.   0.   1.9094147345977                      1.9094147345977                      0.3393499905393206                    0.3393499905393206
1 16.2   0.   0.   0.   0.   1.804160358100995                    1.804160358100995                    0.3207652545356344                    0.3207652545356344
1 17     0.   0.   0.   0.   1.698764910841554                    1.698764910841554                    0.3021346851259734                    0.3021346851259734
1 17.2   0.   0.   0.   0.   1.593236625242241                    1.593236625242241                    0.2834609441545716                    0.2834609441545716
1 18     0.   0.   0.   0.   1.487583745680359                    1.487583745680359                    0.2647466996769285                    0.2647466996769285
1 18.2   0.   0.   0.   0.   1.381814527746618                    1.381814527746618                    0.2459946255759477                    0.2459946255759477
1 19     0.   0.   0.   0.   1.275937237502927                    1.275937237502927                    0.2272074011772387                    0.2272074011772387
1 19.2   0.   0.   0.   0.   1.169960150739085                    1.169960150739085                    0.2083877108636329                    0.2083877108636329
1 20     0.   0.   0.   0.   1.063891552228426                    1.063891552228426                    0.1895382436889724                    0.1895382436889724
1 20.2   0.   0.   0.   0.   0.957739734982543                    0.957739734982543                    0.1706616929912263                    0.1706616929912263
1 21     0.   0.   0.   0.   0.851512999505115                    0.851512999505115                    0.1517607560049894                    0.1517607560049894
1 21.2   0.   0.   0.   0.   0.7452196530449604                   0.7452196530449604                   0.1328381334734208                    0.1328381334734208
1 22     0.   0.   0.   0.   0.6388680088483731                   0.6388680088483731                   0.113896529259677                     0.113896529259677
1 22.2   0.   0.   0.   0.   0.5324663854108244                   0.5324663854108244                   0.0949386499578952                    0.0949386499578952
1 23     0.   0.   0.   0.   0.4260231057281078                   0.4260231057281078                   0.0759672045037843                    0.0759672045037843
1 23.2   0.   0.   0.   0.   0.3195464965470143                   0.3195464965470143                   0.05698490378487808                   0.05698490378487808
1 24     0.   0.   0.   0.   0.2130448876156095                   0.2130448876156095                   0.03799446025050754                   0.03799446025050754
1 24.2   0.   0.   0.   0.   0.1065266109331878                   0.1065266109331878                   0.01899858752154832                   0.01899858752154832
1 25     0.   0.   0.   0.   0.00000000000000001110223024625156   0.00000000000000001110223024625156   0.0000000000000000971445146547012     0.0000000000000000971445146547012;
var I := [*,:,*]
      :1   2 :=
1 25   0.   0.;
var bez_ctls := 
[1,1,*,:]
   :0                     1                    2                    3                     4   5 :=
0   0.   0.04000000000000001   0.1132116860496588   0.1288730162973232   0.04000000000000001   0.
1   0.   0.04000000000000001   0.1132116860496588   0.1288730162973233   0.04000000000000001   0.
2   0.   0.04000000000000001   0.1132116860496587   0.1288730162973233   0.04000000000000001   0.
3   0.   0.04000000000000001   0.1132116860496587   0.1288730162973232   0.04000000000000001   0.;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* seed!
model;

# Amplify Script
We now present a user-defined script.  The core features are the presolve and solver settings and the problem statement.  The problem statement is key as it sets the variables and constraints that will be active during a solve command.  The use of the problem statement is what allows Amplify to be treated as a library.  It acts like an import statement allowing the user to only use what they want from the model definition.

In general, the code below would be generated by an external tool and not by hand.

If the script was executed inside a stand-alone AMPL environment (e.g., the CLI or IDE), the `reset;` statement would be followed by statements to load the model (.mod) and data (.dat) files.  For example,

```ampl
reset;
model myfile.mod;
data myfile.dat;
```

In [ ]:
%%ampl_eval
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* fivelink test
#----------- gait problem
problem FIVELINKTEST :
  UHZD,
  t, q, v, a, u, f, I,
  bez_tau, bez_ctls,
  g_des, gdot_des, gddot_des,
  PHC_CLCK, PHC_Q, PHC_V, PHC_A,
  VHC_CLCK, VHC_Q, VHC_V, VHC_A,
  {i1 in PHASES, i2 in CON_P[i1], i3 in GRID[i1]: i2 <= 2} CONA[i1, i2, i3],
  {i1 in PHASES, i2 in CON_V[i1], i3 in GRID[i1]} CONA[i1, i2, i3],
  {i1 in PHASES, i2 in CON_P[i1], i3 in GRID[i1]: i2 <= 2 and i3 = 0} (CONQ[i1, i2, i3], CONV[i1, i2, i3]),
  {i1 in PHASES, i2 in CON_V[i1], i3 in GRID[i1]: i3 = 0} (CONQ[i1, i2, i3], CONV[i1, i2, i3]),
  EOM,
  ODE_hK,
  ODE_DEFSTEP,
  ODE_DEFQ, ODE_COLQ, ODE_DEFV, ODE_COLV,
  ODE_DEFTIME, ODE_COLTIME,
  T0, TF,
  STEP_LENGTH, STEP_HEIGHT,
  WALKING_SPEED,
  THETA_DOT,
  SWING_FOOT_HEIGHT, GROUND_CLEARANCE,
  STANCE_POSITION_AT_0[1],
  #STANCE_POSITION_AT_0, STANCE_VELOCITY_AT_0,
  SWING_IMPACT_VELX, SWING_IMPACT_VELZ,
  BEZIER_COEFFS,
  PERIODICITY_CONSTRAINT_Q, PERIODICITY_CONSTRAINT_IME,
  PYRAMID_POINT_PLANAR_LB, PYRAMID_POINT_PLANAR_UB, NORMAL_FORCE,
  V_BNDS, A_BNDS, FC_BNDS, U_BNDS,
  F_ZERO, U_ZERO,
  TORSO_PX, TORSO_PZ, TORSO_RY, HIPS, KNEES
;

#----------- solve
solve FIVELINKTEST;
#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#*#* fivelink test!

In [ ]:
%%ampl_eval
print {j in 1.._nvars : match(_varname[j], '^t\[') > 0} : j, _varname[j], _var[j] > test.out;
print {j in 1.._nvars : match(_varname[j], '^x\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^q\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^v\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^a\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^u\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^f\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^I\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^g_des\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^gdot_des\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^gddot_des\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^bez_tau\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^bez_ctls\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^ODE_hK\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^vhc_v\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^phc_v\[') > 0} : j, _varname[j], _var[j] >> test.out;
print {j in 1.._nvars : match(_varname[j], '^bez_curve\[') > 0} : j, _varname[j], _var[j] >> test.out;
close test.out;